# Building the Gemini vector index

Phases 13–16 established that `gemini-embedding-001` beats the incumbent Cohere `embed-v4.0` on this corpus: **11–2 on paired ranks (p = 0.022)** under production's hybrid configuration, at every RRF weight including Cohere's own optimum, and worth **+0.093 MRR** — roughly seven times every fusion parameter combined. This notebook builds the index that makes the switch real.

## Why this costs almost nothing

The 531 corpus vectors were already computed and cached in `eval/emb_cache/` during Phase 13, keyed by a hash of the corpus text. The corpus has not changed since, so **the hash still matches and no re-embedding is needed** — this notebook loads those exact vectors and upserts them.

That is the payoff from caching the expensive artefact rather than only the scores, and it means the switch costs Pinecone writes only.

## What changes

| | before | after |
|---|---|---|
| index | `jordan-labor-law-articles` | `jordan-labor-law-articles-gemini` |
| dimension | 1536 | **3072** |
| query embedding | Cohere `embed-v4.0` | Gemini `gemini-embedding-001` |

A **new** index is created rather than overwriting: dimension differs so an in-place upsert is impossible anyway, and keeping the Cohere index intact means rollback is two constants in `src/retrieval.py` — the same discipline used when Phase 12a switched to article chunking.

## A side effect worth noting

The old index still holds the **watermarked** text for `labor96_ar_37` and `labor96_ar_65`, because the Phase-1 cleanup was never re-indexed. These cached Gemini vectors were computed *after* that cleanup, so building this index **also closes that pending item** — the new index is clean from the start.

## Two known properties of Gemini, carried in deliberately

1. **It silently truncates at 2,048 tokens.** Measured: 4 of 531 articles exceed it (`ssl_ar_71`, `ssl_ar_64`, `ssl_en_71`, `ssl_en_64`), and the API returns a vector with no error rather than complaining. Only SSL 64 is a ground-truth answer for any eval question (q15), which Gemini ranks first regardless. The vectors cached here already carry that truncation, so the index reflects exactly what was evaluated.
2. **Rate limits are a deployment risk, not an indexing one.** Indexing uses no Gemini calls at all. Query-time embedding does, and the free tier's request limits need checking against live traffic before this goes public — the same failure mode that removed Voyage from the Phase 13 candidate list.

In [1]:
import hashlib
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec

PROJECT = Path(r"c:\Users\Mohannad Alsaleh\Desktop\Data Science\Data Science Projects\1st RAG project")
CACHE = PROJECT / "eval" / "emb_cache"
load_dotenv(PROJECT / ".env")

MODEL_NAME = "gemini-embedding-001"
INDEX_NAME = "jordan-labor-law-articles-gemini"
EMBED_DIM = 3072
OLD_INDEX = "jordan-labor-law-articles"

chunks = pd.read_parquet(PROJECT / "0. cleaned_data" / "chunks_articles.parquet")
chunks["article_number"] = chunks["article_number"].astype(str)
chunks["clause"] = chunks["clause"].fillna("").astype(str)

CORPUS_HASH = hashlib.md5(
    "".join(chunks["chunk_id"] + "\x00" + chunks["text"]).encode()).hexdigest()[:12]
VEC_PATH = CACHE / f"{MODEL_NAME}__{CORPUS_HASH}__corpus.npy"

# If the corpus had changed, the hash would differ and this file would not exist --
# which is the point of hashing it. A missing file means re-embed, never silently
# index vectors that describe different text.
assert VEC_PATH.exists(), f"no cached vectors for the current corpus: {VEC_PATH.name}"
vecs = np.load(VEC_PATH)

assert vecs.shape == (len(chunks), EMBED_DIM), f"shape mismatch: {vecs.shape}"
norms = np.linalg.norm(vecs, axis=1)
assert np.allclose(norms, 1.0, atol=1e-3), "vectors are not L2-normalised"
assert not np.isnan(vecs).any(), "NaNs in cached vectors"

# the watermark must be gone from what we are about to index
assert not chunks.text.str.contains("قسطاس", regex=False).any(), "watermark still present"

print(f"corpus hash {CORPUS_HASH}")
print(f"{len(chunks)} articles, vectors {vecs.shape}, all unit-norm, watermark absent")

corpus hash 88dbec8c163f
531 articles, vectors (531, 3072), all unit-norm, watermark absent


## Create the index

Serverless, AWS `us-east-1`, cosine — matching the existing index's configuration so nothing else about the setup changes.

In [2]:
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
existing = [ix["name"] for ix in pc.list_indexes()]
print("indexes before:", existing)

if INDEX_NAME in existing:
    print(f"'{INDEX_NAME}' already exists -- reusing it, not recreating")
else:
    pc.create_index(name=INDEX_NAME, dimension=EMBED_DIM, metric="cosine",
                    spec=ServerlessSpec(cloud="aws", region="us-east-1"))
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        time.sleep(2)
    print(f"created '{INDEX_NAME}' (dim {EMBED_DIM}, cosine)")

index = pc.Index(INDEX_NAME)
print(index.describe_index_stats())

indexes before: ['jordan-labor-law-articles', 'jordan-labor-law']


created 'jordan-labor-law-articles-gemini' (dim 3072, cosine)


DescribeIndexStatsResponse(dimension=3072, total_vector_count=0, metric='cosine', namespaces=0)


## Upsert

Same metadata schema as the Cohere index. `Retriever` reads chunk text from the local parquet rather than from metadata, so these fields are for debuggability and for anything that queries Pinecone directly — not load-bearing for retrieval.

In [3]:
BATCH = 100
rows = []
for i, r in enumerate(chunks.itertuples(index=False)):
    rows.append({
        "id": r.chunk_id,
        "values": vecs[i].tolist(),
        "metadata": {
            "law": r.law,
            "language": r.language,
            "article_number": r.article_number,
            "clause": r.clause,
            "text": r.text,
            "source_file": r.source_file,
        },
    })

for i in range(0, len(rows), BATCH):
    batch = rows[i:i + BATCH]
    for attempt in range(5):
        try:
            index.upsert(vectors=batch)
            break
        except Exception as e:
            wait = min(30, 2 ** attempt)
            print(f"  upsert retry {attempt + 1} at {i}: {type(e).__name__} -- {str(e)[:120]} "
                  f"-- waiting {wait}s")
            time.sleep(wait)
    else:
        raise RuntimeError(f"upsert failed at batch {i}")
    print(f"  upserted {min(i + BATCH, len(rows))}/{len(rows)}", end="\r")

# Pinecone serverless has a short indexing-consistency lag after a bulk upsert
# (noted in Phase 5) -- poll rather than assert immediately.
for _ in range(30):
    n = index.describe_index_stats()["total_vector_count"]
    if n >= len(chunks):
        break
    time.sleep(2)
print(f"\n{index.describe_index_stats()}")
assert n == len(chunks), f"expected {len(chunks)} vectors, index reports {n}"


DescribeIndexStatsResponse(dimension=3072, total_vector_count=531, metric='cosine', namespaces=1)


## Verify against the cache

The strongest available check: query the live index with a *cached query vector* and confirm the article it returns is the one the offline Phase 15/16 evaluation said it would. If these disagree, something is wrong with the upsert — wrong vectors, wrong ids, or a dimension mismatch that Pinecone accepted silently.

In [4]:
ev = pd.DataFrame([json.loads(l) for l in open(PROJECT / "eval" / "eval_v3.jsonl", encoding="utf-8")])
QH = hashlib.md5("".join(ev["question_id"] + "\x00" + ev["question_text"]).encode()).hexdigest()[:12]
qvecs = np.load(CACHE / f"{MODEL_NAME}__{QH}__query.npy")

# offline top-1 from the cached vectors
offline_top = (qvecs @ vecs.T).argmax(axis=1)

agree = 0
for qi in [0, 5, 14, 29, 44, 59]:
    res = index.query(vector=qvecs[qi].tolist(), top_k=1, include_metadata=False)
    live_id = res["matches"][0]["id"]
    off_id = chunks.iloc[offline_top[qi]]["chunk_id"]
    ok = live_id == off_id
    agree += ok
    print(f"  {ev.question_id[qi]}: live={live_id:<18} offline={off_id:<18} {'OK' if ok else 'MISMATCH'}")
assert agree == 6, "live index disagrees with the cached vectors it was built from"
print("\nlive index reproduces the offline evaluation exactly")

  q01: live=labor96_en_86      offline=labor96_en_86      OK


  q06: live=labor96_en_94      offline=labor96_en_94      OK


  q15: live=ssl_en_64          offline=ssl_en_64          OK


  q30: live=ssl_ar_67          offline=ssl_ar_67          OK


  q45: live=labor96_ar_65      offline=labor96_ar_65      OK


  q60: live=prof19_ar_6        offline=prof19_ar_6        OK

live index reproduces the offline evaluation exactly


## Next

`src/retrieval.py` is updated separately to point at this index and to embed queries with Gemini. Rollback remains cheap: switch `PINECONE_INDEX_NAME` and the embedder back to Cohere — the 1536-dim index is untouched and still holds all 531 vectors.

Still outstanding before this is live to the public: **verify Gemini's request limits against real traffic**, and set `GEMINI_API_KEY` in the Railway environment.